# Explore the openmed-deidentification Cloud Run endpoint

Hits the deployed service directly over HTTP -- no `openmed` package install needed here, just `httpx`. Covers: health checks, a single de-identify call, PII extraction, a small batch loop, and a pointer to the interactive `/docs` page (open, no API key needed).

The API key is never hardcoded below -- it's prompted for with `getpass` so it doesn't end up committed in this notebook's output.

In [ ]:
# !pip install httpx

In [ ]:
import getpass
import httpx

BASE_URL = "https://openmed-deidentification-628896201179.us-central1.run.app"
MODEL = "OpenMed/privacy-filter-multilingual-v2"

API_KEY = getpass.getpass("OPENMED_API_KEY: ")

client = httpx.Client(
    base_url=BASE_URL,
    headers={"X-API-Key": API_KEY},
    timeout=180.0,  # CPU-only inference on this model can take 40+ seconds per request
)
print(f"Configured client for {BASE_URL}")

## Health checks

`/health` and `/readyz` don't require the API key -- neither does `/docs` if you want to browse the interactive schema in a browser: [{BASE_URL}/docs]

In [ ]:
print(client.get("/health").json())
print(client.get("/readyz").json())
print(f"Interactive docs: {BASE_URL}/docs")

## Single de-identify request

Expect this to take roughly 30-50 seconds -- CPU-only inference for a 1.4B-parameter MoE model, not a hung request. See the latency note in the repo README.

In [ ]:
text = """
RINGKASAN PULANG PASIEN (SYNTHETIC)

Pasien atas nama Muhammad Rizky Prasetyo, jenis kelamin laki-laki, lahir di Bandung pada 17 Februari 1989 (umur 37 tahun), dengan NIK 3273011702890001, Nomor Rekam Medis (MRN) RM-2026-00182736, dan Nomor SEP 0301R0010826V000123 datang ke RS Harapan Nusantara pada 31 Juli 2026 pukul 08.15 WIB melalui Instalasi Gawat Darurat. Alamat pasien tercatat di Jl. Cendrawasih No. 18, RT 004/RW 009, Kelurahan Sukamaju, Kecamatan Cimahi Tengah, Kota Cimahi, Jawa Barat 40525. Nomor telepon yang dapat dihubungi adalah +62 812-4567-8910, sedangkan email pasien adalah [rizky.prasetyo89@example.com](mailto:rizky.prasetyo89@example.com). Nomor BPJS Kesehatan yang digunakan adalah 0001456789012, nomor polis asuransi tambahan POL-AXA-2026-889912, serta NPWP 12.345.678.9-401.000.

Pasien datang dengan keluhan demam tinggi sejak tiga hari sebelum masuk rumah sakit disertai batuk produktif, sesak napas ringan, dan nyeri dada kanan saat inspirasi. Pasien mengatakan telah mengonsumsi parasetamol 500 mg secara mandiri namun keluhan belum membaik. Riwayat penyakit terdahulu meliputi hipertensi dan diabetes melitus tipe 2 yang dikontrol dengan Amlodipine 10 mg sekali sehari dan Metformin 500 mg tiga kali sehari. Pasien menyangkal riwayat alergi obat.

Pasien dirawat di ruang Mawar Lantai 3 Kamar 305 Bed B dengan Nomor Registrasi Kunjungan REG-20260731-998812. Dokter Penanggung Jawab Pelayanan (DPJP) adalah dr. Andini Kusuma, Sp.P(K) dengan SIP 503/DPMPTSP/SIP-DOK/2025/88231, dibantu oleh dokter jaga dr. Fajar Nugroho. Konsulen penyakit dalam adalah dr. Rendra Wibisono, Sp.PD, sedangkan konsulen radiologi adalah dr. Monica Hartanti, Sp.Rad. Perawat primer yang menangani pasien adalah Ns. Diah Permatasari, S.Kep.

Hasil pemeriksaan awal menunjukkan tekanan darah 150/92 mmHg, nadi 108 kali per menit, suhu tubuh 39,2°C, respirasi 24 kali per menit, saturasi oksigen 94% menggunakan udara ruangan. Pemeriksaan laboratorium menunjukkan leukosit 17.200/µL, CRP meningkat menjadi 112 mg/L, prokalsitonin 3,4 ng/mL, hemoglobin 13,5 g/dL, trombosit 280.000/µL, ureum 28 mg/dL, kreatinin 1,0 mg/dL, SGOT 32 U/L, SGPT 30 U/L, natrium 138 mmol/L, kalium 4,1 mmol/L. Foto toraks menunjukkan infiltrat pada lapang paru kanan bawah yang konsisten dengan pneumonia lobaris.

Diagnosis masuk adalah Pneumonia Komunitas (ICD-10: J18.9) disertai Diabetes Mellitus Tipe 2 (E11.9) dan Hipertensi Esensial (I10). Pasien mendapatkan terapi Ceftriaxone 2 gram intravena setiap 24 jam, Azithromycin 500 mg intravena setiap 24 jam, nebulisasi salbutamol bila diperlukan, terapi cairan NaCl 0,9%, oksigen nasal kanul 2 liter per menit, serta pengendalian gula darah menggunakan insulin sliding scale.

Selama proses administrasi dilakukan verifikasi identitas menggunakan KTP elektronik nomor 3273011702890001, kartu BPJS nomor 0001456789012, dan kartu identitas perusahaan EMP-00892177 milik PT Maju Sejahtera Digital. Penanggung jawab pasien adalah istrinya Siti Rahmawati, lahir pada 5 Mei 1991, dengan NIK 3273010505910002, alamat Jl. Cendrawasih No. 18, RT 004/RW 009, Kelurahan Sukamaju, Kecamatan Cimahi Tengah, Kota Cimahi, nomor telepon 0813-9988-7766, email [siti.rahmawati@example.com](mailto:siti.rahmawati@example.com).

Kontak darurat kedua adalah ayah pasien H. Agus Prasetyo, nomor telepon 0812-1111-2233, alamat Jl. Rajawali No. 7, Bandung 40135. Seluruh persetujuan tindakan medis ditandatangani menggunakan identitas tersebut pada tanggal 31 Juli 2026 pukul 09.45 WIB.

Selama rawat inap dilakukan CT Scan Thorax dengan nomor permintaan RAD-2026-778899, pemeriksaan laboratorium dengan nomor spesimen LAB-20260731-556677, serta kultur sputum dengan nomor sampel MIC-2026-334455. Seluruh pemeriksaan dikirim atas nama pasien Muhammad Rizky Prasetyo dengan nomor rekam medis RM-2026-00182736.

Pada hari kedua perawatan pasien menunjukkan perbaikan klinis. Demam mulai turun, saturasi oksigen meningkat menjadi 98% tanpa bantuan oksigen, leukosit menurun menjadi 11.400/µL. DPJP memutuskan terapi antibiotik dapat dilanjutkan secara oral setelah pasien pulang.

Pasien dipulangkan pada 3 Agustus 2026 pukul 14.20 WIB dengan kondisi stabil. Obat pulang meliputi Levofloxacin 750 mg sekali sehari selama lima hari, Paracetamol 500 mg bila demam, Metformin 500 mg tiga kali sehari, serta Amlodipine 10 mg sekali sehari. Pasien diminta kontrol ke Poli Penyakit Dalam pada 10 Agustus 2026 menggunakan Nomor Antrean POLI-INT-20260810-0045.

Proses klaim menggunakan Nomor SEP 0301R0010826V000123, Nomor Klaim CLM-2026-99887766, Nomor INA-CBG I-4-17-I, dan Kode Billing BILL-20260803-11223344. Total biaya perawatan sebesar Rp18.457.000, ditanggung BPJS sebesar Rp17.930.000, sedangkan selisih Rp527.000 dibayarkan menggunakan rekening BCA 1234567890123 atas nama Muhammad Rizky Prasetyo. Bukti pembayaran dikirim ke email [finance.rizky@example.com](mailto:finance.rizky@example.com).

Pada log sistem rumah sakit tercatat pengguna dr.andini.kusuma, nurse.diah, dan kasir01 mengakses rekam medis pasien. Login dilakukan dari alamat IP 10.10.3.44, 10.10.3.52, dan 192.168.100.25 dengan Session ID SID-6f3c2d8a91be4477a1f0e2, Request ID REQ-20260803-9988776655, dan Device ID DEV-HIS-00441122. Sistem PACS menyimpan pemeriksaan dengan Accession Number ACC-20260731-998712, Study Instance UID 1.2.840.113619.2.55.3.604688.20260731.9987, serta Series Instance UID 1.2.840.113619.2.55.3.604688.20260731.9987.1.

Dokumen administrasi juga mencantumkan data dokter lain yang ikut menangani pasien, yaitu dr. Rendy Saputra, Sp.An, dr. Intan Maharani, Sp.PK, dan dr. Hendra Setiawan, Sp.Rad, lengkap dengan nomor SIP masing-masing 503/SIP/2025/10021, 503/SIP/2025/10022, dan 503/SIP/2025/10023. Selain itu terdapat nomor surat kontrol SK-CTRL-20260810-8877, nomor surat eligibilitas rawat inap SEP-0301R0010826V000123, nomor surat rujukan RJK-2026-55667788, nomor e-resep ERX-9988776655, nomor e-klaim EKLM-4455667788, nomor antrian farmasi FAR-20260803-120, serta barcode internal BC-RM202600182736.

Dokumen ini secara sengaja mengandung berbagai kategori PHI/PII untuk menguji sistem redaksi, termasuk nama pasien, nama keluarga, nama dokter, DPJP, NIK, tanggal lahir, alamat lengkap, nomor telepon, email, nomor rekam medis, nomor SEP, nomor BPJS, nomor polis, nomor SIP dokter, nomor registrasi, nomor spesimen laboratorium, nomor accession radiologi, nomor billing, nomor klaim, nomor rekening, alamat IP, session ID, device ID, identifier internal rumah sakit, lokasi ruang rawat, jadwal kontrol, dan berbagai identifier administratif lainnya. Seluruh data bersifat sintetis dan tidak merepresentasikan individu nyata.

[1]: https://pmc.ncbi.nlm.nih.gov/articles/PMC12926592/?utm_source=chatgpt.com "ASQ-PHI: An adversarial synthetic data benchmark for clinical de-identification and search utility - PMC"
"""

response = client.post(
    "/pii/deidentify",
    json={"text": text, "method": "mask", "model_name": MODEL},
)
response.raise_for_status()
result = response.json()

print("Original:     ", result["original_text"])
print("De-identified:", result["deidentified_text"])
print("Entities redacted:", result["num_entities_redacted"])

## PII extraction (entities + spans, no redaction)

In [ ]:
response = client.post(
    "/pii/extract",
    json={"text": text, "lang": "en", "use_smart_merging": True, "model_name": MODEL},
)
response.raise_for_status()

for entity in response.json()["entities"]:
    print(f"{entity['label']:<15} [{entity['start']:>3}:{entity['end']:<3}]  {entity['text']!r}  conf={entity['confidence']:.3f}")

## Small batch loop

For anything larger than a handful of documents, use `client/batch_client.py` from the repo root instead -- concurrent, retried, writes results incrementally to a JSONL file. This cell is just for quick exploration in the notebook.

Each request is slow (see above), so a handful of texts here can take a couple of minutes sequentially.

In [ ]:
sample_texts = [
    "Patient Jordan Ramirez, MRN 4482910, called from 555-0147.",
    "Paciente: Maria Garcia, correo maria.garcia@example.com, tel 555-0199.",
]

for i, sample in enumerate(sample_texts, start=1):
    response = client.post(
        "/pii/deidentify",
        json={"text": sample, "method": "mask", "model_name": MODEL},
    )
    response.raise_for_status()
    print(f"[{i}/{len(sample_texts)}] {response.json()['deidentified_text']}")

## Next steps

- Real batch processing: `python ../client/batch_client.py --base-url {BASE_URL} --api-key $OPENMED_API_KEY --input ../examples/sample_input.jsonl --output results.jsonl`
- Throughput benchmarking against this exact endpoint: `python ../client/benchmark_rest.py --base-url {BASE_URL} --api-key $OPENMED_API_KEY --concurrency-levels 1,2,4,8`
- Full endpoint reference: `{BASE_URL}/docs`